In [2]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')


import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

import random

# Funzioni utilizzate

In [2]:
######################CLUSTERING######################

import networkx as nx

def ClusterComponentiConnessi(MatchTable, TuttiInodi):

    MatchTable=deepcopy(MatchTable)
    MatchTable.columns=['A','B']

    Singleton = set(TuttiInodi) - set(MatchTable['A']).union(set(MatchTable['B']))

    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['A'], row['B'])
#        G.add_edge(row['A'], row['B'], weight=row['sim'])  # Aggiungi il peso (etichetta) basato su 'sim'

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [3]:
def VisualizzaCluster2(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

def VisualizzaCluster(Clusters,quanti):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI DELL'IDENTIFICATIVO DEL RECORD
### specificere in quanti ; esempio se il record è S2_123, quanti =2 per ottenere S2
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[:quanti]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


def _VisualizzaDistribuzioneCluster(Clusters):
    gruppi = Clusters.groupby('ClusterKey')
    conteggio_gruppi = gruppi.size().reset_index(name='NumeroElementiPerCluster')
    Risultato=conteggio_gruppi.groupby('NumeroElementiPerCluster').size().reset_index(name='NumeroCluster')
    print("Numero Elementi", (Risultato['NumeroElementiPerCluster'] * Risultato['NumeroCluster']).sum())
    ClusterMax=conteggio_gruppi[conteggio_gruppi['NumeroElementiPerCluster']==Risultato['NumeroElementiPerCluster'].max()]
    print("Cluster con max numero di elementi:", ClusterMax['ClusterKey'].tolist())

    return Risultato

In [4]:
def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

In [5]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [6]:
def Valuta2(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta2(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [7]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
 #   Gold = Gold[['l_id','r_id']]
 #   Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    #Gold = Gold[['l_id','r_id']]
    #Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [8]:
def ValutaBlocking(DA,DB,Block,Gold):
# INPUT : entrambi Block (è il candidate set after blocking)
#        e Gold (Gold Standard) devono essere con due colonne, l_id e r_id
# per avere indipendenza dal nome di queste colonne
# Si suppone che in Block l_id e r_id siano rispettivamente la seconda e la terza colonna
# e che in Gold sia la prima e la seconda
  Block = Block.iloc[:, [1, 2]].copy()
  Gold = Gold.iloc[:, [0, 1]].copy()
  Gold.columns=Block.columns=['l_id','r_id']

  JOIN=pd.merge(Gold, Block)

 # Reduction_Ratio
  RR=1-len(Block)/(len(DA)*len(DB))
 # Pairs Completeness o Recall
  PC = len(JOIN)/len(Gold)
 # Pairs Quality
  PQ = len(JOIN)/len(Block)

  Risultato = pd.DataFrame([(DA.shape[0],DB.shape[0],Block.shape[0],round(RR,4),round(PC,4),round(PQ,4))],
                             columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

  return Risultato

In [9]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

# Esercizio Entity Resolution

In [1]:
path='http://dbgroup.ing.unimore.it/EBI/Cluster/'


src_links = [
path+'A.csv',
path+'B.csv',
path+'C.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

ClusterGoldStandard=pd.read_csv(path+ 'ClusterGoldStandard.csv')

def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS
VisualizzaCluster(ClusterGoldStandard).sort_values('#Record', ascending=False)

NameError: name 'pd' is not defined

In [15]:
SOURCES['S2']

,Nome,Cognome,DataNascita,Sesso,Nazionalita,CodiceBelfiore,id
0,Khalid,Scccah,05/15/1996,M,Marocco,Z330,B_0
1,Amadeo,Borsellino,02/24/1966,M,Italia,Z112,B_1
2,Giorgivo,Mazza,04/24/1988,M,Italia,G479,B_2
3,Gianfranco,Sardagna,06/04/1999,M,Italia,G479,B_3
4,Francesco,Casolari,06/23/1987,M,Italia,C134,B_4
...,...,...,...,...,...,...,...
614,Andriy,Shafraniuccc,09/11/1988,M,Ucraina,Z138,B_614
615,Gheorhe,Moldoveanu,12/13/1978,M,Romania,Z129,B_615
616,Francesco Filippo,Bizzoni,08/14/1973,M,Italia,G282,B_616
617,Arsne Kra,Konan,12/19/1999,M,Costa D'Avorio,Z313,B_617


In [13]:
df = VisualizzaCluster(ClusterGoldStandard).sort_values('#Record', ascending=False)
df = df[df['#Sorgenti'] < df['#Record']]
df
# clean dataset

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record


In [16]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 1875
Cluster con max numero di elementi: [5, 66, 69, 88, 93]


,NumeroElementiPerCluster,NumeroCluster
0,1,1300
1,2,280
2,3,5


In [26]:

UNIONE = pd.DataFrame()

for x in SOURCES.keys():
    UNIONE=UNIONE.append(SOURCES[x])
          
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['Nome_Nome_jac_qgm_3_qgm_3',
 'Nome_Nome_cos_dlm_dc0_dlm_dc0',
 'Nome_Nome_jac_dlm_dc0_dlm_dc0',
 'Nome_Nome_mel',
 'Nome_Nome_lev_dist',
 'Nome_Nome_lev_sim',
 'Nome_Nome_nmw',
 'Nome_Nome_sw',
 'Cognome_Cognome_jac_qgm_3_qgm_3',
 'Cognome_Cognome_cos_dlm_dc0_dlm_dc0',
 'Cognome_Cognome_jac_dlm_dc0_dlm_dc0',
 'Cognome_Cognome_mel',
 'Cognome_Cognome_lev_dist',
 'Cognome_Cognome_lev_sim',
 'Cognome_Cognome_nmw',
 'Cognome_Cognome_sw',
 'DataNascita_DataNascita_lev_dist',
 'DataNascita_DataNascita_lev_sim',
 'DataNascita_DataNascita_jar',
 'DataNascita_DataNascita_jwn',
 'DataNascita_DataNascita_exm',
 'DataNascita_DataNascita_jac_qgm_3_qgm_3',
 'Sesso_Sesso_lev_dist',
 'Sesso_Sesso_lev_sim',
 'Sesso_Sesso_jar',
 'Sesso_Sesso_jwn',
 'Sesso_Sesso_exm',
 'Sesso_Sesso_jac_qgm_3_qgm_3',
 'Nazionalita_Nazionalita_jac_qgm_3_qgm_3',
 'Nazionalita_Nazionalita_cos_dlm_dc0_dlm_dc0',
 'Nazionalita_Nazionalita_jac_dlm_dc0_dlm_dc0',
 'Nazionalita_Nazionalita_mel',
 'Nazionalita_Nazionalita_lev_dist

In [105]:
# Metodo di Entity Resolution dato
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    # BLOCKING by Similarity Join
    Attributi = ['Nome', 'Cognome','Sesso', 'Nazionalita', 'CodiceBelfiore']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    CandidateDato  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                        l_out_attrs=Attributi,
                                        r_out_attrs=Attributi
                                     )
    CandidateDato=CandidateDato.rename(columns={'l_l_id': 'l_id'})
    CandidateDato=CandidateDato.rename(columns={'r_r_id': 'r_id'})
    CandidateDato=CandidateDato.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)
 #   print(ValutaBlocking(A,B,CandidateDato[['l_id','r_id']],GoldStandard[['l_id','r_id']]))


############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING
    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['Nome_Nome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 +Cognome_Cognome_jac_qgm_3_qgm_3(ltuple, rtuple)*0.7 > 0.7', 'Sesso_Sesso_exm(ltuple,rtuple) == 1'], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

            MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
            MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
            #MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

            MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [106]:
MTSOURCES=MatchTableSOURCES(SOURCES)
MTSOURCES


,l_id,r_id,sim
0,A_584,B_214,1.000000
1,A_257,B_218,1.000000
2,A_119,B_283,1.000000
3,A_446,B_185,1.000000
4,A_449,B_455,1.000000
...,...,...,...
107,B_546,C_474,0.384615
108,B_461,C_443,0.369565
109,B_481,C_69,0.333333
110,B_228,C_80,0.326923


In [107]:
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 1875
Cluster con max numero di elementi: [20, 48, 49, 64, 73, 84, 89, 95, 100]


,NumeroElementiPerCluster,NumeroCluster
0,1,1232
1,2,308
2,3,9


In [108]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,335,294,41,1,0.8776,0.9966,0.9333


In [109]:
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,Nome_x,Cognome_x,DataNascita_x,Sesso_x,Nazionalita_x,CodiceBelfiore_x,id_x,Nome_y,Cognome_y,DataNascita_y,Sesso_y,Nazionalita_y,CodiceBelfiore_y,id_y
0,B_267,C_253,right_only,Giacinto,Maffei,01/22/1970,M,Italia,Z110,B_267,Umberto,Maffei,25-02-1956,M,Italia,Z133,C_253
1,A_469,C_253,right_only,Giacinto,Maffei,22/01/1970,M,Italia,Z110,A_469,Umberto,Maffei,25-02-1956,M,Italia,Z133,C_253
2,B_220,C_411,right_only,Amadeo,Guicciardini,03/14/1967,M,Italia,H199,B_220,Marcello,Guicciardini,20-07-1963,M,Italia,G713,C_411
3,A_585,C_411,right_only,Amaseo,Guicciardini,14/03/1967,M,Italia,H199,A_585,Marcello,Guicciardini,20-07-1963,M,Italia,G713,C_411
4,A_206,B_368,right_only,Franco,Venturini,04/02/1981,M,Italia,F205,A_206,Marco,Venturini,09/26/1983,M,Italia,E573,B_368
5,A_310,B_470,right_only,Aurelio,Pietrangeli,14/02/1974,M,Italia,E783,A_310,Giacinto,Pietrangeli,02/10/1966,M,Italia,B990,B_470
6,A_480,B_129,right_only,Francesco,D'Aniello,08/05/1968,M,Italia,H892,A_480,Enrico,D'Aniello,12/18/1992,M,Italia,F205,B_129
7,B_129,C_622,right_only,Enrico,D'Aniello,12/18/1992,M,Italia,F205,B_129,Francesco,D'Aniello,08-05-1968,M,Italia,H892,C_622
8,A_308,B_326,right_only,Saverio,Santorio,10/07/1997,M,Italia,D607,A_308,Enrico,Santorio,04/23/1972,M,Italia,E063,B_326
9,A_43,B_333,right_only,Arturo,Morucci,30/12/1965,M,Italia,D018,A_43,Alderano,Morucci,08/10/1990,M,Italia,M403,B_333


In [94]:
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,Nome_x,Cognome_x,DataNascita_x,Sesso_x,Nazionalita_x,CodiceBelfiore_x,id_x,Nome_y,Cognome_y,DataNascita_y,Sesso_y,Nazionalita_y,CodiceBelfiore_y,id_y
0,A_327,B_15,left_only,Gianfranco,Pabdolfini,15/10/1968,M,Italia,E335,A_327,Gianfranco,Pandolfini,10/15/1968,M,Italia,E335,B_15
1,A_167,C_546,left_only,Nikolat Ivanovich,Paslar,24/04/1994,M,Bulgaria,Z104,A_167,Nikolay,Paslar,24-04-1994,M,Bulgaria,Z104,C_546
2,B_26,C_546,left_only,Nikolay Ivanovich,Paslar,04/24/1994,M,Bulgaria,Z104,B_26,Nikolay,Paslar,24-04-1994,M,Bulgaria,Z104,C_546
3,A_608,B_44,left_only,Hichan,Al-Siguni,26/10/1984,M,Marocco,Z330,A_608,Hicham,Al-Siguni,10/26/1984,M,Marocco,Z330,B_44
4,A_510,B_49,left_only,Giorgio,Pennacchietti,03/12/1969,M,Italia,D773,A_510,Giorgio,Pennacchieyti,12/03/1969,M,Italia,D773,B_49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,B_544,C_603,left_only,Giovanni,Benfrateloo,05/23/1987,M,Italia,E290,B_544,Giovanni,Benfratello,23-05-1987,M,Italia,E290,C_603
105,B_210,C_606,left_only,Samuel Okon,Peter,02/25/1979,M,Nigeria,Z335,B_210,Samuel,Peter,25-02-1979,M,Nigeria,Z335,C_606
106,B_366,C_613,left_only,Egidio,Jacsuzzi,05/04/1975,M,Italia,D379,B_366,Egidio,Jacuzzi,04-05-1975,M,Italia,D379,C_613
107,B_21,C_619,left_only,Nicol,Delli Santi,08/13/2004,M,Italia,D810,B_21,Nicola,Delli Santi,13-08-2004,M,Italia,D810,C_619


Discussione

Lo svolgimento/risposta consiste nella discussione strutturata nei seguenti punti

* Visualizzare la distribuzione dei cluster del ClusterGoldStandard dato

*    Considerando il Metodo di Entity Resolution dato, calcolare i cluster ottenuti e quindi la valutazione rispetto a quelli dati da ClusterGoldStandard in termini di Match Indotti

*    Considerando i FP e/o FN che si ottengono, modificare il Metodo di Entity Resolution dato (sia nella funzione BlockingMatchingRule che nella funzione MatchTableSOURCES dove si decide se usare o meno un global mapping) e discutere miglioramenti
